In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_val = pd.read_csv("../data/processed/X_val.csv")

y_train = pd.read_csv("../data/processed/y_train.csv")
y_val = pd.read_csv("../data/processed/y_val.csv")

y_train = y_train.values.ravel()
y_val = y_val.values.ravel()

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

Train shape: (2966, 12)
Validation shape: (636, 12)


In [3]:
def evaluate_model(name, model):

    model.fit(X_train, y_train)

    pred_prob = model.predict_proba(X_val)[:,1]
    pred = model.predict(X_val)

    roc = roc_auc_score(y_val, pred_prob)
    precision = precision_score(y_val, pred)
    recall = recall_score(y_val, pred)

    print("Model:", name)
    print("ROC-AUC:", round(roc,4))
    print("Precision:", round(precision,4))
    print("Recall:", round(recall,4))
    print("-"*40)

    return roc, precision, recall, model

In [4]:
log_model = LogisticRegression(max_iter=1000)

log_roc, log_precision, log_recall, log_model = evaluate_model(
    "Logistic Regression",
    log_model
)

Model: Logistic Regression
ROC-AUC: 0.9657
Precision: 0.9935
Recall: 0.7251
----------------------------------------


In [5]:
dt_model = DecisionTreeClassifier(
    max_depth=8,
    random_state=42
)

dt_roc, dt_precision, dt_recall, dt_model = evaluate_model(
    "Decision Tree",
    dt_model
)

Model: Decision Tree
ROC-AUC: 0.9368
Precision: 0.8722
Recall: 0.7441
----------------------------------------


In [6]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf_roc, rf_precision, rf_recall, rf_model = evaluate_model(
    "Random Forest",
    rf_model
)

Model: Random Forest
ROC-AUC: 0.9664
Precision: 0.963
Recall: 0.7393
----------------------------------------


In [7]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_roc, xgb_precision, xgb_recall, xgb_model = evaluate_model(
    "XGBoost",
    xgb_model
)

Model: XGBoost
ROC-AUC: 0.966
Precision: 0.8859
Recall: 0.7725
----------------------------------------


In [8]:
nn_model = MLPClassifier(
    hidden_layer_sizes=(64,32),
    max_iter=300,
    random_state=42
)

nn_roc, nn_precision, nn_recall, nn_model = evaluate_model(
    "Neural Network",
    nn_model
)

Model: Neural Network
ROC-AUC: 0.9657
Precision: 0.861
Recall: 0.763
----------------------------------------


c:\Users\shali\ecommerce-churn-prediction\venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [9]:
pickle.dump(log_model, open("../models/logistic_regression.pkl","wb"))
pickle.dump(dt_model, open("../models/decision_tree.pkl","wb"))
pickle.dump(rf_model, open("../models/random_forest.pkl","wb"))
pickle.dump(xgb_model, open("../models/xgboost_model.pkl","wb"))
pickle.dump(nn_model, open("../models/neural_network.pkl","wb"))

print("All models saved")

All models saved


In [10]:
scores = {
    "Logistic": log_roc,
    "DecisionTree": dt_roc,
    "RandomForest": rf_roc,
    "XGBoost": xgb_roc,
    "NeuralNetwork": nn_roc
}

best_model_name = max(scores, key=scores.get)

print("Best Model:", best_model_name)

Best Model: RandomForest


In [11]:
best_model = {
    "Logistic": log_model,
    "DecisionTree": dt_model,
    "RandomForest": rf_model,
    "XGBoost": xgb_model,
    "NeuralNetwork": nn_model
}[best_model_name]

pickle.dump(best_model, open("../models/best_model.pkl","wb"))

print("Best model saved")

Best model saved


In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost", "Neural Network"],
    "ROC_AUC": [log_roc, dt_roc, rf_roc, xgb_roc, nn_roc],
    "Precision": [log_precision, dt_precision, rf_precision, xgb_precision, nn_precision],
    "Recall": [log_recall, dt_recall, rf_recall, xgb_recall, nn_recall],
})

results.to_csv("../data/processed/model_comparison.csv", index=False)
results

,Model,ROC_AUC,Precision,Recall
0,Logistic,0.965732,0.993506,0.725118
1,DecisionTree,0.936777,0.872222,0.744076
2,RandomForest,0.966367,0.962963,0.739336
3,XGBoost,0.965999,0.885870,0.772512
4,NeuralNetwork,0.965743,0.860963,0.763033


: 